## Full MIMIC-IV: TPC remaining ICU LoS (verification notebook)

This notebook mirrors `examples/tpc_mimic4_full_mimic_train.py` so you can **inspect every intermediate step** before committing to a long run.

**What you can verify**

- Paths to your local MIMIC-IV v3.x tree (`hosp/`, `icu/`)
- Table 17 label strings → `itemid` resolution via `d_labitems.csv.gz` / `d_items.csv.gz` (including ambiguous matches)
- `MIMIC4EHRDataset` initialization and cache locations
- Task-bound sample counts and (after caches exist) a **single processed batch** (tensor shapes)
- Optional **full training** + masked test metrics, including **binned “accuracy”** (exact match on `categorize_los` buckets; regression primary metrics are MAE/MSE/MSLE)

**Performance**

- Full `labevents` / `chartevents` scans are **slow** the first time while `global_event_df.parquet` is built. Start with `DEV_MODE = True` and `NO_CHARTEVENTS = True`, then scale up.

**CSV ingest**

- If PyArrow’s streaming CSV reader fails (sometimes surfaced as `OSError: Truncated compressed stream`), this repo’s `BaseDataset._scan_csv_tsv_gz` **falls back to Polars** automatically. **§5a** (subprocess pre-warm) is optional extra isolation—enable it in the config cell only if you still need it.


In [ ]:
# --- User configuration (edit these) ---

# Root folder that contains hosp/ and icu/ (MIMIC-IV v3.1 layout).
EHR_ROOT = "../datasets/mimic-iv/3.1"

# Where PyHealth will write dataset + task caches (use a dedicated folder per experiment).
CACHE_DIR = None  # default: <repo>/.pyhealth_dataset_cache_full_nb

# Safer defaults for first verification runs.
DEV_MODE = True  # MIMIC4EHRDataset dev=True caps patients (~1000) — NOT full cohort.
NO_CHARTEVENTS = True  # labs-only: excludes chartevents table (much faster I/O).

# Optional: materialize `global_event_df` in a plain `python` subprocess (§5a).
# PyHealth now falls back to Polars for CSV→Parquet when PyArrow streaming fails, so
# this is usually unnecessary—set True only if you still want subprocess isolation.
WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS = False

# Training (increase after pipeline checks out).
EPOCHS = 1
BATCH_SIZE = 4
LR = 0.00221
WEIGHT_DECAY = 0.0
MONITOR = "mae"  # "mae" | "mse"


### 0) Repo imports + cache path

Run from `examples/` or repo root; we always add the repo root to `sys.path` so you load **this** `pyhealth`, not a site-packages install.


In [ ]:
import importlib.util
import os
import sys

import numpy as np
import polars as pl
import torch
from IPython.display import display

# Resolve REPO_ROOT robustly whether cwd is repo root or examples/
_here = os.path.abspath(os.getcwd())
if os.path.basename(_here) == "examples":
    REPO_ROOT = os.path.abspath(os.path.join(_here, ".."))
else:
    REPO_ROOT = _here

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

os.environ.setdefault("PYHEALTH_CACHE_PATH", os.path.join(REPO_ROOT, ".pyhealth_cache"))

if CACHE_DIR is None:
    CACHE_DIR = os.path.join(REPO_ROOT, ".pyhealth_dataset_cache_full_nb")

print("REPO_ROOT:", REPO_ROOT)
print("PYHEALTH_CACHE_PATH:", os.environ["PYHEALTH_CACHE_PATH"])
print("CACHE_DIR:", CACHE_DIR)
print("torch:", torch.__version__)


REPO_ROOT: /home/kevinchen101/PyHealth-TPC_LoS-AKV
PYHEALTH_CACHE_PATH: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_cache
CACHE_DIR: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb
torch: 2.7.1+cu126


### 1) Sanity-check MIMIC-IV on-disk layout


In [ ]:
required = [
    os.path.join(EHR_ROOT, "hosp", "patients.csv.gz"),
    os.path.join(EHR_ROOT, "hosp", "admissions.csv.gz"),
    os.path.join(EHR_ROOT, "icu", "icustays.csv.gz"),
    os.path.join(EHR_ROOT, "hosp", "labevents.csv.gz"),
    os.path.join(EHR_ROOT, "hosp", "d_labitems.csv.gz"),
    os.path.join(EHR_ROOT, "icu", "d_items.csv.gz"),
]

optional_chart = os.path.join(EHR_ROOT, "icu", "chartevents.csv.gz")

missing = [p for p in required if not os.path.isfile(p)]
assert not missing, f"Missing required files:\n" + "\n".join(missing)

print("OK: required core files exist.")
print("chartevents present:", os.path.isfile(optional_chart), optional_chart)

if not NO_CHARTEVENTS:
    assert os.path.isfile(optional_chart), "NO_CHARTEVENTS is False but chartevents.csv.gz not found."


OK: required core files exist.
chartevents present: True ../datasets/mimic-iv/3.1/icu/chartevents.csv.gz


### 2) Load helper constants + functions from `tpc_mimic4_full_mimic_train.py`

This avoids copy-pasting Table 17 lists and keeps metric definitions identical to the script.


In [ ]:
# Use a dedicated module name and register in sys.modules *before* exec_module.
# Python 3.13's @dataclass resolves forward refs via sys.modules[cls.__module__];
# exec_module without registration leaves that lookup as None and crashes.

_MOD_NAME = "pyhealth_nb_full_mimic_train_helpers"
_script_path = os.path.join(REPO_ROOT, "examples", "tpc_mimic4_full_mimic_train.py")
spec = importlib.util.spec_from_file_location(_MOD_NAME, _script_path)
ft = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[_MOD_NAME] = ft
spec.loader.exec_module(ft)

TABLE17_CHART_LABELS = ft.TABLE17_CHART_LABELS
TABLE17_LAB_LABELS = ft.TABLE17_LAB_LABELS
resolve_itemids_from_dictionary = ft.resolve_itemids_from_dictionary
compute_seq_metrics = ft.compute_seq_metrics

print("Loaded:", _script_path, "as", _MOD_NAME)
print("Chart labels:", len(TABLE17_CHART_LABELS), "Lab labels:", len(TABLE17_LAB_LABELS))


Loaded: /home/kevinchen101/PyHealth-TPC_LoS-AKV/examples/tpc_mimic4_full_mimic_train.py as pyhealth_nb_full_mimic_train_helpers
Chart labels: 58 Lab labels: 45


### 3) Inspect dictionary tables (Polars schema / row counts)

Confirms `infer_schema_length=100_000` read works for `d_items` / `d_labitems` (mixed numeric columns).


In [ ]:
d_items_path = os.path.join(EHR_ROOT, "icu", "d_items.csv.gz")
d_lab_path = os.path.join(EHR_ROOT, "hosp", "d_labitems.csv.gz")

d_items_raw = pl.read_csv(d_items_path, infer_schema_length=100_000)
d_lab_raw = pl.read_csv(d_lab_path, infer_schema_length=100_000)

print("d_items:", d_items_raw.shape)
print(d_items_raw.head(3))
print("d_labitems:", d_lab_raw.shape)
print(d_lab_raw.head(3))


d_items: (4095, 9)
shape: (3, 9)
┌────────┬────────────┬────────────┬────────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ itemid ┆ label      ┆ abbreviati ┆ linksto    ┆ … ┆ unitname ┆ param_typ ┆ lownormal ┆ highnorma │
│ ---    ┆ ---        ┆ on         ┆ ---        ┆   ┆ ---      ┆ e         ┆ value     ┆ lvalue    │
│ i64    ┆ str        ┆ ---        ┆ str        ┆   ┆ str      ┆ ---       ┆ ---       ┆ ---       │
│        ┆            ┆ str        ┆            ┆   ┆          ┆ str       ┆ i64       ┆ f64       │
╞════════╪════════════╪════════════╪════════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 220001 ┆ Problem    ┆ Problem    ┆ chartevent ┆ … ┆ null     ┆ Text      ┆ null      ┆ null      │
│        ┆ List       ┆ List       ┆ s          ┆   ┆          ┆           ┆           ┆           │
│ 220003 ┆ ICU        ┆ ICU        ┆ datetimeev ┆ … ┆ null     ┆ Date and  ┆ null      ┆ null      │
│        ┆ Admission  ┆ Admission  ┆ ents       ┆   ┆     

### 4) Resolve Table 17 labels → `itemid` (and list ambiguities / misses)

The training script uses **deterministic tie-break**: if multiple rows share the same normalized label, pick the **smallest** `itemid`.


In [ ]:
if NO_CHARTEVENTS:
    chart_itemids, lab_itemids, amb = resolve_itemids_from_dictionary(
        ehr_root=EHR_ROOT,
        chart_labels=[],
        lab_labels=TABLE17_LAB_LABELS,
    )
    print("Mode: labs-only (chartevents not used).")
else:
    chart_itemids, lab_itemids, amb = resolve_itemids_from_dictionary(
        ehr_root=EHR_ROOT,
        chart_labels=TABLE17_CHART_LABELS,
        lab_labels=TABLE17_LAB_LABELS,
    )
    print("Mode: chartevents + labevents.")

print("Resolved chart itemids:", len(chart_itemids), "/", len(TABLE17_CHART_LABELS))
print("Resolved lab itemids:  ", len(lab_itemids), "/", len(TABLE17_LAB_LABELS))
print("Ambiguous labels (multiple hits):", len(amb))

list(amb.items())[:5]


Mode: labs-only (chartevents not used).
Resolved chart itemids: 0 / 58
Resolved lab itemids:   43 / 45
Ambiguous labels (multiple hits): 28


[('MCHC', ['51249', '53185']),
 ('Alkaline Phosphatase', ['50863', '53086']),
 ('MCV', ['51250', '51691']),
 ('Anion Gap', ['50868', '52500']),
 ('Base Excess', ['50802', '52038'])]

In [ ]:
# Per-label resolution table for labs (always) + charts (if used)


def _norm_label(s: str) -> str:
    return " ".join(s.strip().split()).casefold()


def _resolve_table(table: pl.DataFrame, labels: list[str]) -> pl.DataFrame:
    t = table.select(pl.col("itemid").cast(pl.Int64), pl.col("label").cast(pl.Utf8))
    t = t.with_columns(pl.col("label").map_elements(_norm_label, return_dtype=pl.Utf8).alias("label_n"))

    rows = []
    for lab in labels:
        ln = _norm_label(lab)
        hits = t.filter(pl.col("label_n") == ln)
        if hits.height == 0:
            rows.append({"paper_label": lab, "chosen_itemid": None, "n_hits": 0, "all_itemids": None})
        else:
            iids = sorted(hits["itemid"].to_list())
            rows.append(
                {
                    "paper_label": lab,
                    "chosen_itemid": str(iids[0]) if len(iids) else None,
                    "n_hits": hits.height,
                    "all_itemids": ",".join(str(x) for x in iids[:12]) + ("..." if len(iids) > 12 else ""),
                }
            )
    return pl.DataFrame(rows)


d_lab_small = pl.read_csv(d_lab_path, infer_schema_length=100_000).select(
    pl.col("itemid"),
    pl.col("label"),
)
lab_map_df = _resolve_table(d_lab_small, TABLE17_LAB_LABELS)
lab_map_df.sort("n_hits", descending=True).head(15)


paper_label,chosen_itemid,n_hits,all_itemids
str,str,i64,str
"""pH""","""50820""",6,"""50820,50831,51094,51491,52041,…"
"""Glucose""","""50809""",5,"""50809,50931,51478,51981,52569"""
"""Hematocrit""","""51221""",5,"""51221,51480,51638,51639,52028"""
"""Potassium""","""50833""",3,"""50833,50971,52610"""
"""Hemoglobin""","""50811""",3,"""50811,51222,51640"""
…,…,…,…
"""Anion Gap""","""50868""",2,"""50868,52500"""
"""Base Excess""","""50802""",2,"""50802,52038"""
"""PT""","""51274""",2,"""51274,52921"""


In [ ]:
missed_labs = lab_map_df.filter(pl.col("chosen_itemid").is_null())
print("Unresolved lab labels:", missed_labs.height)
missed_labs


Unresolved lab labels: 2


paper_label,chosen_itemid,n_hits,all_itemids
str,str,i64,str
"""Time in the ICU""",null,0,null
"""Time of day""",null,0,null


In [ ]:
# Optional: chart label → itemid table (only if chartevents enabled)

if not NO_CHARTEVENTS:
    d_items_small = pl.read_csv(d_items_path, infer_schema_length=100_000).select(
        pl.col("itemid"),
        pl.col("label"),
    )
    chart_map_df = _resolve_table(d_items_small, TABLE17_CHART_LABELS)
    display(chart_map_df.sort("n_hits", descending=True).head(15))
    missed_charts = chart_map_df.filter(pl.col("chosen_itemid").is_null())
    print("Unresolved chart labels:", missed_charts.height)
    display(missed_charts)
else:
    print("Skipped (NO_CHARTEVENTS=True).")


Skipped (NO_CHARTEVENTS=True).


### 5) Build `MIMIC4EHRDataset` (this can take a long time on first run)

Watch stdout for `global_event_df.parquet` creation progress.

If `WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS` is `True` in the config cell, run **§5a** next; otherwise continue to **§6** (`set_task`).


In [ ]:
from pyhealth.datasets import MIMIC4EHRDataset, get_dataloader, split_by_patient
from pyhealth.models import TPC
from pyhealth.tasks import RemainingLengthOfStayTPC_MIMIC4
from pyhealth.trainer import Trainer

tables = ["patients", "admissions", "icustays", "labevents"]
if not NO_CHARTEVENTS:
    tables.append("chartevents")

ds = MIMIC4EHRDataset(
    root=EHR_ROOT,
    tables=tables,
    dev=DEV_MODE,
    num_workers=1,
    cache_dir=CACHE_DIR,
)

print("Dataset tables:", tables)
print("dev:", DEV_MODE)


Using default EHR config: /home/kevinchen101/PyHealth-TPC_LoS-AKV/pyhealth/datasets/configs/mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 2466.0 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from ../datasets/mimic-iv/3.1 (dev mode: True)
Using provided cache_dir: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb/37f667d5-cfc8-5edb-8dbc-50342ff579ca
Memory usage After initializing mimic4_ehr: 2466.0 MB
Dataset tables: ['patients', 'admissions', 'icustays', 'labevents']
dev: True


### 5a) (Optional) Pre-build `global_event_df` in a plain Python subprocess

`BaseDataset._scan_csv_tsv_gz` now **falls back to Polars** when PyArrow’s streaming CSV reader fails, so you can usually **skip this section** and go straight to **§6**.

Use this cell only when `WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS = True` in the config—e.g. if you want the heavy CSV scan to run in a separate `python` process instead of this Jupyter kernel.


In [ ]:
import json
import subprocess
import tempfile

# Safe default if the config cell was not re-run after this flag was added.
if "WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS" not in globals():
    WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS = False
    print("WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS was undefined; defaulting to False.")

if not WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS:
    print("Skipping subprocess pre-warm (WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS=False).")
else:
    assert "ds" in globals(), "Run the previous cell to construct `ds` (and `tables`) first."

    payload = {
        "repo_root": REPO_ROOT,
        "pyhealth_cache": os.environ.get("PYHEALTH_CACHE_PATH", ""),
        "mimic_kwargs": {
            "root": os.path.abspath(EHR_ROOT),
            "tables": tables,
            "dev": bool(DEV_MODE),
            "num_workers": 1,
            "cache_dir": CACHE_DIR,
        },
    }

    runner = r"""import json, os, sys

cfg_path = sys.argv[1]
with open(cfg_path, encoding="utf-8") as f:
    cfg = json.load(f)
if cfg.get("pyhealth_cache"):
    os.environ["PYHEALTH_CACHE_PATH"] = cfg["pyhealth_cache"]
sys.path.insert(0, cfg["repo_root"])

from pyhealth.datasets import MIMIC4EHRDataset

ds = MIMIC4EHRDataset(**cfg["mimic_kwargs"])
# Forces CSV scan + dask write of global_event_df.parquet (outside IPython).
_ = ds.global_event_df
print("subprocess_warm_ok", str(ds.cache_dir))
"""

    with tempfile.NamedTemporaryFile("w", encoding="utf-8", suffix="_warm_mimic.py", delete=False) as sf:
        sf.write(runner)
        script_path = sf.name
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", suffix="_warm_mimic.json", delete=False) as jf:
        json.dump(payload, jf)
        cfg_path = jf.name

    try:
        subprocess.check_call(
            [sys.executable, script_path, cfg_path],
            cwd=REPO_ROOT,
        )
    finally:
        for p in (script_path, cfg_path):
            try:
                os.unlink(p)
            except OSError:
                pass

    # This kernel must not keep a stale in-memory path before reading on-disk cache.
    ds._global_event_df = None  # noqa: SLF001
    print("Notebook `ds` reset; run §6 (`set_task`) next.")


Skipping subprocess pre-warm (WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS=False).


### 5b) If ingest still fails (truncated stream / `unexpected end of file`)

**A — Source `.csv.gz` is corrupt or incomplete (most common when *both* PyArrow and Polars fail)**  
Large MIMIC files must match the official release size. A partial copy often triggers *Truncated compressed stream* (PyArrow) and *unexpected end of file* (Polars).

- In a terminal: `gzip -t /path/to/hosp/labevents.csv.gz` — if this errors, the archive is bad.
- Compare file size to the [MIMIC-IV](https://physionet.org/content/mimiciv/) documentation; **re-download** `hosp/labevents.csv.gz` from PhysioNet if needed.

**B — Stale or half-written PyHealth cache**  
The **`global_event_df.parquet`** folder under `ds.cache_dir` may be incomplete. Restart the kernel, set `CLEAR_GLOBAL_EVENT_CACHE = True` in the next cell, re-run §5, then §6.


In [ ]:
# Manual reset (only if needed). Requires `ds` from the previous cell.
import shutil
from pathlib import Path

assert "ds" in globals(), "Run the MIMIC4EHRDataset cell first."

c = Path(ds.cache_dir)
ged = c / "global_event_df.parquet"
print("cache_dir:", c)
print("global_event_df.parquet exists:", ged.exists())

# Set True to delete the global event cache and force a full rebuild on next access.
CLEAR_GLOBAL_EVENT_CACHE = False

if CLEAR_GLOBAL_EVENT_CACHE and ged.exists():
    shutil.rmtree(ged)
    # Allow the property to rediscover / rebuild
    ds._global_event_df = None  # noqa: SLF001
    print("Removed", ged)


cache_dir: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb/37f667d5-cfc8-5edb-8dbc-50342ff579ca
global_event_df.parquet exists: False


### 6) Attach task + patient-level split


In [ ]:
from pathlib import Path

if "WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS" not in globals():
    WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS = False

if WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS:
    ged = Path(ds.cache_dir) / "global_event_df.parquet"
    if not (ged.is_dir() and any(ged.glob("*.parquet"))):
        raise RuntimeError(
            "Missing or empty global_event_df cache. Run §5a (subprocess pre-warm) before this cell, "
            "or set WARM_GLOBAL_EVENT_CACHE_VIA_SUBPROCESS = False if your environment does not need it."
        )

task = RemainingLengthOfStayTPC_MIMIC4(
    labevent_itemids=lab_itemids,
    chartevent_itemids=chart_itemids,
)

sample_ds = ds.set_task(task)
train_ds, val_ds, test_ds = split_by_patient(sample_ds, ratios=[0.7, 0.15, 0.15])

print("train / val / test:", len(train_ds), len(val_ds), len(test_ds))


Setting task RemainingLengthOfStayTPC_MIMIC4 for mimic4_ehr base dataset...
Task cache paths: task_df=/home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb/37f667d5-cfc8-5edb-8dbc-50342ff579ca/tasks/RemainingLengthOfStayTPC_MIMIC4_5404aa23-2b65-5be0-97d9-b4c6b3bc5a4a/task_df.ld, samples=/home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb/37f667d5-cfc8-5edb-8dbc-50342ff579ca/tasks/RemainingLengthOfStayTPC_MIMIC4_5404aa23-2b65-5be0-97d9-b4c6b3bc5a4a/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache_full_nb/37f667d5-cfc8-5edb-8dbc-50342ff579ca/global_event_df.parquet
Scanning table: admissions from /home/kevinchen101/PyHealth-TPC_LoS-AKV/datasets/mimic-iv/3.1/hosp/admissions.csv.gz
Scanning table: patients from /home/kevinchen101/PyHealth-TPC_LoS-AKV/datasets/mimic-iv/3.1/hosp/patients.csv.gz


OSError: Truncated compressed stream

### 7) Inspect **one processed sample** (tensor shapes / keys)

This indexes the task dataset, so it runs the processors for a single patient sample.


In [ ]:
s0 = sample_ds[0]
print("Top-level keys:", sorted(s0.keys()))
for k, v in s0.items():
    if hasattr(v, "shape"):
        print(k, type(v), getattr(v, "shape", None), v.dtype if hasattr(v, "dtype") else None)
    else:
        print(k, type(v))

# Light assertions you can tighten while debugging
assert "ts" in s0 and "static" in s0 and "y" in s0
assert s0["ts"].ndim == 3, s0["ts"].shape  # (T, F, 2)
assert s0["y"].ndim == 1, s0["y"].shape


### 8) Peek **one training batch** (batched shapes)


In [ ]:
train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print("Batch keys:", sorted(batch.keys()))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)


### 9) Instantiate `TPC` + run `Trainer` (long-running)

After training, we run `trainer.inference` on the test loader and print **masked** metrics:
`mae`, `mse`, `msle`, `bin_accuracy`, `cohen_kappa_bins`.


In [ ]:
model = TPC(
    dataset=sample_ds,
    temporal_channels=11,
    pointwise_channels=5,
    num_layers=8,
    kernel_size=5,
    main_dropout=0.0,
    temporal_dropout=0.05,
    use_batchnorm=True,
    final_hidden=36,
)

trainer = Trainer(model, metrics=["mae", "mse"], enable_logging=True)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    epochs=EPOCHS,
    optimizer_class=torch.optim.Adam,
    optimizer_params={"lr": LR},
    weight_decay=WEIGHT_DECAY,
    monitor=MONITOR,
    monitor_criterion="min",
    load_best_model_at_last=True,
)


In [ ]:
y_true_all, y_pred_all, _loss = trainer.inference(test_loader)

mask = y_true_all != 0
extra = compute_seq_metrics(y_true=y_true_all, y_pred=y_pred_all, mask=mask)

print("\n=== Additional test metrics (sequence-level, masked) ===")
for k in sorted(extra.keys()):
    print(f"{k}: {extra[k]:.6f}")

extra
